# Plot saved Yeo2011 ROI correlations

Load the annotated CSV produced by `correlations.ipynb` and plot **every observed
`model_type` × `step` combination** for the selected field, metric, and outcome. No model or step
list is hard-coded. This notebook needs no variables or functions from another
running notebook, and it does not rerun correlations or extract raw fields.

Two plot families are enabled by default:

| Family | Figure contents |
|---|---|
| Whole-atlas overview | All seven parents together: one regional-ROI map and one bilateral-network map per configuration |
| Parent-network plots | One parent per figure: separate LH/RH regional coefficients, then its single combined-hemisphere network coefficient |

**Regional** means each individual LH/RH ROI has its own rho. **Combined network**
means the correlation was computed from the per-series average of that parent's
regional field means across both hemispheres. Its one saved rho fills every member
region. ROI correlations are never averaged to create a network coefficient.

All plot sets use the saved rho values, a shared -1 to 1 color scale, the atlas
geometry, and the matching FSL MNI152 background. No p-value mask is applied.
Each set has a descriptive matching-prefix NIfTI, PNG, and source-subset CSV.
Run all cells when ready; figures are saved and closed to avoid accumulating
open windows. The optional preview reads saved PNGs.

Feature identity uses explicit `field_name` and `metric` columns. For example, `magnE_mean_7Networks_LH_DorsAttn_FEF` preserves source field `magnE_mean`; `E_7Networks_LH_DorsAttn_FEF` preserves the different source field `E`. The field prefix is matched literally, including any underscores.


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import image, plotting

field_name = "magnE_mean"
metric = "mean"
outcome = "cgi_change"

correlations_path = Path(
    "/projects/p32903/Alex2/ect-workflow/05_analysis/networks/data"
) / f"spearman_{field_name}_ROImean_and_network_annotated.csv"
atlas_path = Path(
    "/projects/p32903/Alex2/ect-workflow/atlases/Yeo2011/"
    "Yeo2011_7Networks_N1000.split_components.FSL_MNI152_1mm.nii.gz"
)
lut_path = atlas_path.parent / "7Networks_ColorLUT_freeview.txt"
template_path = Path(
    "/projects/p32903/Alex2/simnibs_dev/simnibs/simnibs/"
    "resources/templates/MNI152_T1_1mm.nii.gz"
)

# Field-specific destination avoids mixing these maps with another field's run.
output_dir = correlations_path.parent / f"spearman_{field_name}_plots"

plot_overviews = True
plot_parents = True
networks = None                 # None = all atlas parents; or ["Default", "Cont"].

# Preview one parent's saved maps for every configuration after generation.
show_preview = False
preview_network = "Default"

## Functions

Image construction and per-parent plotting reuse the original notebook. The wrappers load saved coefficients, iterate the observed configurations, save overviews, and record output paths.

In [2]:
def annotate_correlation_features(results: pd.DataFrame) -> pd.DataFrame:
    """Parse feature labels using the exact source field, never underscore guesses."""
    required = {"x", "field_name", "metric"}
    missing = required - set(results.columns)
    if missing:
        raise KeyError(f"Missing feature identity columns: {sorted(missing)}")

    result = results.copy()
    parents = "Vis|SomMot|DorsAttn|SalVentAttn|Limbic|Cont|Default"
    metadata = []

    for feature, source_field, metric_name in result[["x", "field_name", "metric"]].itertuples(index=False, name=None):
        if not isinstance(source_field, str) or not source_field or source_field != source_field.strip():
            raise ValueError(f"Invalid source field_name: {source_field!r}")
        if not isinstance(metric_name, str) or not metric_name:
            raise ValueError(f"Invalid ROI summary metric: {metric_name!r}")

        prefix = f"{source_field}_"
        if not isinstance(feature, str) or not feature.startswith(prefix):
            raise ValueError(f"Feature {feature!r} does not preserve field_name={source_field!r}.")

        label = feature[len(prefix):]
        regional = re.fullmatch(rf"7Networks_(LH|RH)_({parents})(?:_(.+))?", label)
        bilateral = re.fullmatch(rf"({parents})_both_hemis", label)

        if regional:
            hemi, network, component = regional.groups()
            metadata.append((label, component or network, hemi, network))
        elif bilateral:
            network = bilateral.group(1)
            metadata.append((None, network, "both", network))
        else:
            raise ValueError(f"Unrecognized ROI/network label in feature {feature!r}.")

    parsed = pd.DataFrame(metadata, columns=["roi_org", "roi", "hemi", "network"], index=result.index)
    for column in parsed:
        if column in result.columns:
            agrees = result[column].fillna("").eq(parsed[column].fillna(""))
            if not agrees.all():
                raise ValueError(f"Existing {column!r} metadata conflicts with x and field_name.")
        result[column] = parsed[column]

    return result


def load_scalar_img(niimg):
    """Load a 3D image, accepting a singleton fourth dimension."""
    img = image.load_img(niimg)

    if len(img.shape) == 4 and img.shape[-1] == 1:
        img = img.slicer[..., 0]

    if len(img.shape) != 3:
        raise ValueError(f"Expected 3D or (X,Y,Z,1); got {img.shape}")

    return img


def load_label_atlas(atlas_path, lut_path):
    """Return the atlas and a Series mapping label IDs to ROI names."""
    atlas = load_scalar_img(atlas_path)
    ids = np.unique(image.get_data(atlas))

    if not np.isfinite(ids).all() or not np.equal(ids, np.rint(ids)).all():
        raise ValueError("The atlas must contain finite integer labels.")

    ids = ids[ids != 0].astype(int)

    lut = pd.read_csv(
        lut_path,
        sep=r"\s+",
        comment="#",
        header=None,
        usecols=[0, 1],
        names=["id", "name"],
    ).set_index("id")["name"]

    if not lut.index.is_unique:
        raise ValueError("Duplicate IDs in the LUT.")

    names = lut.reindex(ids)

    if names.empty or names.isna().any() or not names.is_unique:
        raise ValueError("Every atlas ROI needs a unique name in the LUT.")

    return atlas, names


def network_correlations_to_image(results, atlas_path, lut_path, *, field_name: str = "magnE_mean", metric: str = "mean", level="network"):
    """Map either 'roi' or 'network' correlations onto the atlas."""
    if level not in ("roi", "network"):
        raise ValueError("level must be 'roi' or 'network'.")

    results = results.loc[
        results["field_name"].eq(field_name) & results["metric"].eq(metric)
    ].copy()
    if results.empty:
        raise ValueError(f"No coefficients for field_name={field_name!r}, metric={metric!r}.")
    results = annotate_correlation_features(results)

    if results["x"].duplicated().any():
        raise ValueError("Duplicate features: select one configuration only.")

    if results["y"].nunique(dropna=False) != 1:
        raise ValueError("Results must contain exactly one outcome.")

    atlas, names = load_label_atlas(atlas_path, lut_path)
    labels = image.get_data(atlas)

    networks = names.str.extract(
        r"^7Networks_(?:LH|RH)_([^_]+)(?:_|$)",
        expand=False,
    )

    if networks.isna().any():
        raise ValueError(
            f"Unrecognized atlas labels: {names[networks.isna()].tolist()}"
        )

    # Each Series maps atlas label IDs to correlation feature names.
    roi_features = f"{field_name}_" + names
    network_features = f"{field_name}_" + networks + "_both_hemis"

    known_features = set(roi_features) | set(network_features)
    unexpected = set(results["x"]) - known_features

    if unexpected:
        raise ValueError(f"Unrecognized features: {sorted(unexpected)}")

    features = roi_features if level == "roi" else network_features
    required = set(features)

    selected = results.loc[results["x"].isin(required)]
    missing = required - set(selected["x"])

    if missing:
        raise ValueError(f"Missing {level} features: {sorted(missing)}")

    rho = pd.to_numeric(
        selected.set_index("x")["rho"],
        errors="raise",
    )

    if not np.isfinite(rho.to_numpy(dtype=float)).all():
        raise ValueError("Correlation coefficients contain non-finite values.")

    if rho.abs().gt(1).any():
        raise ValueError("Correlation coefficients must be within [-1, 1].")

    rho_volume = np.zeros(labels.shape, dtype=np.float32)

    for label_id, feature in features.items():
        rho_volume[labels == label_id] = rho.loc[feature]

    rho_img = image.new_img_like(atlas, rho_volume, copy_header=True)
    rho_img.set_data_dtype(np.float32)

    return rho_img


def plot_save_parent_network(results, *, network, model_type, step, atlas_path, lut_path, template_path, output_dir, field_name: str = "magnE_mean", metric: str = "mean", y="cgi_change"):
    """Save regional and bilateral-network maps for one parent/configuration."""
    atlas, names = load_label_atlas(atlas_path, lut_path)
    labels = image.get_data(atlas)

    # Select this parent's atlas regions, keeping both hemispheres.
    roi_names = names.loc[
        names.str.split("_").str[2].eq(network)
    ]

    hemispheres = set(roi_names.str.split("_").str[1])
    if roi_names.empty or hemispheres != {"LH", "RH"}:
        raise ValueError(
            f"Atlas must contain LH and RH regions for {network!r}."
        )

    roi_features = (f"{field_name}_" + roi_names).tolist()
    bilateral_feature = f"{field_name}_{network}_both_hemis"
    required = [*roi_features, bilateral_feature]

    # Select one configuration and outcome.
    selected = results.loc[
        results["model_type"].eq(model_type)
        & results["step"].eq(step)
        & results["y"].eq(y)
        & results["field_name"].eq(field_name)
        & results["metric"].eq(metric)
    ].copy()

    selected = annotate_correlation_features(selected)

    selected = selected.loc[selected["x"].isin(required)]

    if selected["x"].duplicated().any():
        duplicates = selected.loc[
            selected["x"].duplicated(keep=False), "x"
        ].tolist()
        raise ValueError(f"Duplicate correlation features: {duplicates}")

    missing = set(required) - set(selected["x"])
    if missing:
        raise ValueError(
            f"{model_type}/{step}/{network}: "
            f"missing correlation features: {sorted(missing)}"
        )

    selected = selected.set_index("x").loc[required].copy()
    rho = pd.to_numeric(selected["rho"], errors="raise")

    if not np.isfinite(rho.to_numpy(dtype=float)).all():
        raise ValueError("Non-finite correlation coefficients.")

    if rho.abs().gt(1).any():
        raise ValueError("Correlation coefficients must be within [-1, 1].")

    selected["rho"] = rho

    parent_mask = np.isin(labels, roi_names.index)
    regional_volume = np.zeros(labels.shape, dtype=np.float32)

    # Choose a representative axial slice for every component ROI.
    # Both map types use these same deterministic slice coordinates.
    cuts = []

    for label_id, roi_name in roi_names.items():
        roi_mask = labels == label_id
        regional_volume[roi_mask] = rho.loc[f"{field_name}_{roi_name}"]

        k = int(roi_mask.sum(axis=(0, 1)).argmax())
        i, j = np.argwhere(roi_mask[:, :, k])[0]
        world_z = (atlas.affine @ [i, j, k, 1])[2]
        cuts.append(float(world_z))

    cuts = sorted(set(cuts))

    bilateral_volume = np.zeros(labels.shape, dtype=np.float32)
    bilateral_volume[parent_mask] = rho.loc[bilateral_feature]

    # Transparency hides everything outside this parent network,
    # while allowing a genuine rho=0 region to remain visible.
    mask_img = image.new_img_like(
        atlas,
        parent_mask.astype(np.uint8),
        copy_header=False,
    )
    background = image.load_img(template_path)

    jobs = [
        (
            "regional_ROIs_LH_RH_separate",
            regional_volume,
            roi_features,
            "Each LH/RH region has its own correlation coefficient",
        ),
        (
            "network_mean_both_hemis",
            bilateral_volume,
            [bilateral_feature],
            "One network coefficient: regional field means combined across LH + RH",
        ),
    ]

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    saved = {}

    for tag, volume, features, description in jobs:
        prefix = (
            f"spearman_rho_field-{field_name}_metric-{metric}_{model_type}_{step}_{y}_"
            f"parent_{network}_{tag}"
        )

        paths = {
            "nifti": output_dir / f"{prefix}.nii.gz",
            "png": output_dir / f"{prefix}.png",
            "csv": output_dir / f"{prefix}.csv",
        }

        rho_img = image.new_img_like(atlas, volume, copy_header=True)
        rho_img.set_data_dtype(np.float32)
        rho_img.header["cal_min"] = -1
        rho_img.header["cal_max"] = 1
        rho_img.to_filename(paths["nifti"])

        # Regional CSV: one row per component ROI.
        # Bilateral CSV: exactly one network-level correlation row.
        selected.loc[features].reset_index().to_csv(
            paths["csv"],
            index=False,
        )

        figure = plt.figure(
            figsize=(max(10, 2 * len(cuts) + 1), 4)
        )

        display = plotting.plot_stat_map(
            rho_img,
            bg_img=background,
            figure=figure,
            display_mode="z",
            cut_coords=cuts,
            threshold=None,
            transparency=mask_img,
            transparency_range=[0, 1],
            resampling_interpolation="nearest",
            cmap="RdBu_r",
            vmin=-1,
            vmax=1,
            symmetric_cbar=True,
            colorbar=True,
            cbar_tick_format="%.2f",
            title=(
                f"Field: {field_name} | metric: {metric} | parent: {network}\n"
                f"{model_type} | {step}\n"
                f"{description}\nSpearman rho vs {y}"
            ),
        )

        try:
            display.savefig(paths["png"], dpi=300)
        finally:
            display.close()

        saved[tag] = paths

    return saved


def load_correlation_results(filename: str | Path) -> pd.DataFrame:
    """Read saved coefficients and validate explicit source-field identities."""
    path = Path(filename)
    if not path.is_file():
        raise FileNotFoundError(f"Correlation CSV does not exist: {path}")
    results = pd.read_csv(path)
    required = {"x", "y", "n", "rho", "p_value", "field_name", "metric", "model_type", "step", "roi_org", "roi", "hemi", "network"}
    missing = required - set(results.columns)
    if missing:
        raise KeyError(f"Missing field-aware correlation columns: {sorted(missing)}")
    if results.empty:
        raise ValueError("Correlation CSV is empty.")
    keys = ["field_name", "metric", "model_type", "step", "y", "x"]
    if results[keys].isna().any().any():
        raise ValueError("Missing source-field, metric, configuration, outcome, or feature identifiers.")
    results = annotate_correlation_features(results)
    if results.duplicated(keys).any():
        raise ValueError("Duplicate field/metric/model/step/outcome/feature correlations.")
    results["rho"] = pd.to_numeric(results["rho"], errors="raise")
    if not np.isfinite(results["rho"].to_numpy(dtype=float)).all():
        raise ValueError("Correlation coefficients contain non-finite values.")
    if results["rho"].abs().gt(1).any():
        raise ValueError("Correlation coefficients must be within [-1, 1].")
    return results


def summarize_correlation_input(results: pd.DataFrame) -> pd.DataFrame:
    """Count saved correlation rows for each available model/step configuration."""
    return (
        results.groupby(["field_name", "metric", "model_type", "step"], sort=False)
        .size()
        .rename("correlation_rows")
        .to_frame()
    )


def plot_save_overviews(results: pd.DataFrame, model_type: str, step: str, atlas_path, lut_path, template_path, output_dir, field_name: str = "magnE_mean", metric: str = "mean", y: str = "cgi_change") -> dict:
    """Save both whole-atlas maps and their exact source correlation subsets."""
    results = results.loc[
        results["model_type"].eq(model_type)
        & results["step"].eq(step)
        & results["y"].eq(y)
        & results["field_name"].eq(field_name)
        & results["metric"].eq(metric)
    ].copy()
    results = annotate_correlation_features(results)
    _, names = load_label_atlas(atlas_path, lut_path)
    parents = names.str.split("_").str[2].drop_duplicates().tolist()
    jobs = [
        (
            "roi",
            "all_networks_regional_ROIs_LH_RH_separate",
            (f"{field_name}_" + names).tolist(),
            "All parent networks: each LH/RH regional ROI has its own rho",
        ),
        (
            "network",
            "all_networks_network_means_both_hemis",
            [f"{field_name}_{parent}_both_hemis" for parent in parents],
            "All parent networks: one bilateral network-mean rho per parent",
        ),
    ]
    background = load_scalar_img(template_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    saved = {}

    for level, tag, features, description in jobs:
        rho_img = network_correlations_to_image(results, atlas_path, lut_path, field_name=field_name, metric=metric, level=level)
        prefix = f"spearman_rho_field-{field_name}_metric-{metric}_{model_type}_{step}_{y}_{tag}"
        paths = {
            "nifti": output_dir / f"{prefix}.nii.gz",
            "png": output_dir / f"{prefix}.png",
            "csv": output_dir / f"{prefix}.csv",
        }
        rho_img.to_filename(paths["nifti"])
        results.set_index("x").loc[features].reset_index().to_csv(paths["csv"], index=False)

        display = plotting.plot_stat_map(
            rho_img,
            bg_img=background,
            display_mode="z",
            cut_coords=[-20, 0, 20, 40, 60],
            threshold=0,
            cmap="RdBu_r",
            vmin=-1,
            vmax=1,
            symmetric_cbar=True,
            resampling_interpolation="nearest",
            colorbar=True,
            cbar_tick_format="%.2f",
            title=f"{field_name} | metric: {metric} | {model_type} | {step}\nSpearman rho vs {y}\n{description}",
        )
        try:
            display.savefig(paths["png"], dpi=300)
        finally:
            display.close()
        saved[tag] = paths

    return saved


def plot_correlation_configurations(results: pd.DataFrame, atlas_path, lut_path, template_path, output_dir, field_name: str = "magnE_mean", metric: str = "mean", y: str = "cgi_change", networks: list | None = None, plot_overviews: bool = True, plot_parents: bool = True) -> pd.DataFrame:
    """Plot all observed configurations and return a manifest of saved files."""
    if not plot_overviews and not plot_parents:
        raise ValueError("Enable at least one plot family.")
    selected = results.loc[
        results["y"].eq(y)
        & results["field_name"].eq(field_name)
        & results["metric"].eq(metric)
    ].copy()
    if selected.empty:
        raise ValueError(f"No saved correlations for field={field_name!r}, metric={metric!r}, outcome={y!r}.")

    _, names = load_label_atlas(atlas_path, lut_path)
    available = names.str.split("_").str[2].drop_duplicates().tolist()
    parents = available if networks is None else list(networks)
    if plot_parents:
        if not parents or len(parents) != len(set(parents)):
            raise ValueError("networks must contain unique parent names.")
        unknown = set(parents) - set(available)
        if unknown:
            raise ValueError(f"Unknown parent networks: {sorted(unknown)}")

    output_dir = Path(output_dir)
    records = []
    for (model_type, step), table in selected.groupby(["model_type", "step"], sort=False, observed=True):
        print(f"Plotting {field_name} / {metric} / {model_type} / {step}", flush=True)

        if plot_overviews:
            saved = plot_save_overviews(
                table,
                model_type=model_type,
                step=step,
                atlas_path=atlas_path,
                lut_path=lut_path,
                template_path=template_path,
                output_dir=output_dir / "overview_maps",
                field_name=field_name,
                metric=metric,
                y=y,
            )
            for tag, paths in saved.items():
                records.append({
                    "field_name": field_name, "metric": metric,
                    "model_type": model_type, "step": step, "network": "all_networks",
                    "family": "overview", "plot_type": tag,
                    **{key: str(value) for key, value in paths.items()},
                })

        if plot_parents:
            for parent in parents:
                saved = plot_save_parent_network(
                    table,
                    network=parent,
                    model_type=model_type,
                    step=step,
                    atlas_path=atlas_path,
                    lut_path=lut_path,
                    template_path=template_path,
                    output_dir=output_dir / "parent_network_maps",
                    field_name=field_name,
                    metric=metric,
                    y=y,
                )
                for tag, paths in saved.items():
                    records.append({
                        "field_name": field_name, "metric": metric,
                        "model_type": model_type, "step": step, "network": parent,
                        "family": "parent", "plot_type": tag,
                        **{key: str(value) for key, value in paths.items()},
                    })

    return pd.DataFrame(records)


def save_plot_manifest(manifest: pd.DataFrame, filename: str | Path) -> Path:
    """Save paths and configuration labels for the generated map sets."""
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(path, index=False)
    return path


def summarize_plot_outputs(manifest: pd.DataFrame) -> pd.DataFrame:
    """Count generated map sets for each observed configuration and family."""
    return (
        manifest.groupby(["field_name", "metric", "model_type", "step", "family"], sort=False)
        .size()
        .unstack("family", fill_value=0)
        .rename_axis(columns=None)
    )


def preview_saved_plots(manifest: pd.DataFrame, network: str = "Default") -> None:
    """Display one parent's saved regional/bilateral PNGs for every configuration."""
    from IPython.display import Image, display

    selected = manifest.loc[
        manifest["family"].eq("parent") & manifest["network"].eq(network)
    ]
    if selected.empty:
        raise ValueError(f"No saved parent-network plots for {network!r}.")
    for row in selected.itertuples(index=False):
        print(f"{row.field_name} / {row.metric} / {row.model_type} / {row.step} / {row.network} / {row.plot_type}")
        display(Image(filename=row.png))

## Load saved correlations

In [3]:
all_results = load_correlation_results(correlations_path)
print(f"Loaded {len(all_results)} saved correlations from {correlations_path}")
# summarize_correlation_input(all_results)

Loaded 232 saved correlations from /projects/p32903/Alex2/ect-workflow/05_analysis/networks/data/spearman_magnE_mean_ROImean_and_network_annotated.csv


## Plot and save every observed configuration

This cell creates the figures and files. Each configuration is handled independently; the coefficients are read directly from the input table.

In [4]:
plot_manifest = plot_correlation_configurations(
    all_results,
    atlas_path=atlas_path,
    lut_path=lut_path,
    template_path=template_path,
    output_dir=output_dir,
    field_name=field_name,
    metric=metric,
    y=outcome,
    networks=networks,
    plot_overviews=plot_overviews,
    plot_parents=plot_parents,
)
print(f"Saved {len(plot_manifest)} NIfTI/PNG/source-CSV map sets under {output_dir}")

Plotting magnE_mean / mean / skin_single / step_0
Plotting magnE_mean / mean / skin_single / step_9
Plotting magnE_mean / mean / skin_double / step_0


: 

## Save the output manifest

In [ ]:
manifest_path = save_plot_manifest(plot_manifest, output_dir / "plot_manifest.csv")
print(f"Saved plot manifest to {manifest_path}")
summarize_plot_outputs(plot_manifest)

## Output interpretation

With the current four configurations and all seven parents enabled, expect
**2 overview sets + 14 parent sets per configuration**, or **64 map sets total**.
Each set has matching-prefix `.nii.gz`, `.png`, and `.csv` files. A separate
`plot_manifest.csv` lists the configuration, parent, plot type, and exact paths.

- `overview_maps/`: both map types show all seven parents together. Filenames
  explicitly distinguish hemisphere-specific regional ROIs from bilateral
  network means. Overview source CSVs contain 51 regional rows or seven
  bilateral network rows. Fixed axial cuts match the original notebook.
- `parent_network_maps/`: one parent per figure. The regional source CSV has
  one row per component ROI (ten for Default); the bilateral CSV has exactly
  one row. Both types use the same deterministic component-based cuts.
- Colors encode raw Spearman rho, not network membership or p-value significance.
  Parent transparency retains actual zero coefficients inside the selected
  network; overview plots use the original zero-value threshold.
- The regional and bilateral correlation rows must both be present, as produced
  by `correlations.ipynb` with `include_network_means=True`. Missing/ambiguous
  coefficients raise errors rather than being silently skipped.
- `networks` limits parent plots only. All observed model/step combinations
  for the selected outcome are discovered from the CSV and plotted.

The input CSV, master table, and original analysis notebook are only read.
Run this notebook yourself when ready; its cells have not been executed here.

## Optional preview

Set `show_preview=True` in the settings cell to display the chosen parent's saved regional and bilateral figures for every configuration.

In [ ]:
if show_preview:
    preview_saved_plots(plot_manifest, network=preview_network)